# **LABORATORIO DE PROCESAMIENTO DIGITAL DE IMAGENES (PDI)**

[Nombre completo integrante 1] - [Matrícula]

[Nombre completo integrante 2] - [Matrícula]

[Nombre completo integrante 3] - [Matrícula]

Brigada: [Número de brigada]

Día y hora: [Día y hora de clase]

# **PRÁCTICA 4. *PROCESAMIENTO GEOMÉTRICO Y PROCESAMIENTO DE HISTOGRAMA***

---


---

**Importación de librerías y carga de las imágenes**

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Cargar la imagen médica para la Parte I (procesamiento geométrico) en escala de grises
img_geo = cv2.imread('/content/radiografia.jpg', cv2.IMREAD_GRAYSCALE)

# Cargar la imagen médica de bajo contraste para la Parte II (ecualización de histograma)
img_hist = cv2.imread('/content/resonancia_bajo_contraste.jpg', cv2.IMREAD_GRAYSCALE)

# Mostrar las dimensiones de ambas imágenes para verificar que se cargaron correctamente
print(f"Dimensiones imagen geométrica: {img_geo.shape}")
print(f"Dimensiones imagen histograma: {img_hist.shape}")

> Se importan las librerías necesarias para la práctica: `cv2` (OpenCV) para las operaciones de procesamiento de imágenes, `numpy` para el manejo de matrices y `matplotlib.pyplot` para la visualización de resultados.

> Ambas imágenes médicas se cargan directamente en escala de grises mediante la bandera `cv2.IMREAD_GRAYSCALE`, ya que las radiografías y resonancias magnéticas suelen representarse en un solo canal de intensidad, lo cual simplifica las transformaciones geométricas y el análisis de histograma que se realizarán más adelante.

# **Parte I. Procesamiento geométrico**

**Ejercicio 1. Traslación**

In [ ]:
# Obtener las dimensiones (filas, columnas) de la imagen
filas, columnas = img_geo.shape

# Matriz de traslación con valores enteros (50, 30)
M_entera = np.float32([[1, 0, 50],
                        [0, 1, 30]])
img_trasladada_entera = cv2.warpAffine(img_geo, M_entera, (columnas, filas))

# Matriz de traslación con valores decimales (20.5, 15.5)
M_decimal = np.float32([[1, 0, 20.5],
                         [0, 1, 15.5]])
img_trasladada_decimal = cv2.warpAffine(img_geo, M_decimal, (columnas, filas))

# Mostrar la imagen original y las dos imágenes trasladadas
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img_geo, cmap='gray')
plt.title('Imagen original')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img_trasladada_entera, cmap='gray')
plt.title('Traslación (50, 30) px')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(img_trasladada_decimal, cmap='gray')
plt.title('Traslación (20.5, 15.5) px')
plt.axis('off')

plt.tight_layout()
plt.show()

> Para trasladar la imagen se construyó la matriz de transformación afín \(2\times3\) que utiliza `cv2.warpAffine`, en donde los valores de la tercera columna (`tx`, `ty`) corresponden al desplazamiento en píxeles sobre los ejes X y Y respectivamente.

> En el primer caso se aplicó un desplazamiento de valores enteros (50, 30), mientras que en el segundo se utilizaron valores decimales (20.5, 15.5). OpenCV resuelve internamente estos corrimientos sub-píxel mediante interpolación (bilineal por defecto), por lo que la imagen trasladada de forma decimal presenta una suavización sutil en los bordes en comparación con la traslación entera.

**Ejercicio 2. Rotación**

In [ ]:
# Definir el centro de la imagen como punto de rotación
centro = (columnas // 2, filas // 2)

# Obtener la matriz de rotación para un ángulo de 45° sin escalado adicional
M_rotacion = cv2.getRotationMatrix2D(centro, 45, 1.0)

# Aplicar la rotación a la imagen
img_rotada = cv2.warpAffine(img_geo, M_rotacion, (columnas, filas))

# Mostrar la imagen original y la imagen rotada
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img_geo, cmap='gray')
plt.title('Imagen original')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_rotada, cmap='gray')
plt.title('Rotación 45°')
plt.axis('off')

plt.tight_layout()
plt.show()

> La rotación se calculó con `cv2.getRotationMatrix2D`, indicando como punto de pivote el centro geométrico de la imagen (`columnas // 2`, `filas // 2`), un ángulo de 45° y un factor de escala de 1.0 (sin modificar el tamaño). Esta función devuelve la matriz de transformación afín correspondiente, la cual se aplica posteriormente con `cv2.warpAffine`.

> Al rotar sobre el centro y mantener el mismo tamaño de lienzo (`columnas`, `filas`), las esquinas de la imagen original quedan recortadas en la imagen resultante, ya que el área rotada excede el marco original.

**Ejercicio 3. Escala**

In [ ]:
# Escalado al 150% del tamaño original
img_escalada_150 = cv2.resize(img_geo, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_LINEAR)

# Escalado al 50% del tamaño original
img_escalada_50 = cv2.resize(img_geo, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)

# Mostrar la imagen original y los dos resultados de escalado
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img_geo, cmap='gray')
plt.title('Imagen original')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img_escalada_150, cmap='gray')
plt.title('Escalado 150%')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(img_escalada_50, cmap='gray')
plt.title('Escalado 50%')
plt.axis('off')

plt.tight_layout()
plt.show()

> El escalado se realizó con `cv2.resize`, indicando los factores `fx` y `fy` en lugar de un tamaño fijo en píxeles, de forma que la relación de aspecto se mantiene proporcional al tamaño original.

> Para el aumento de tamaño (150%) se utilizó el método de interpolación bilineal (`cv2.INTER_LINEAR`), adecuado para ampliar imágenes, mientras que para la reducción (50%) se empleó `cv2.INTER_AREA`, recomendado por OpenCV para el submuestreo ya que reduce el aliasing al promediar los píxeles del área correspondiente.

# **Parte II. Ecualización de histograma**

**Ejercicio 1. Ecualización de histograma**

In [ ]:
# Calcular el histograma de la imagen original de bajo contraste
hist_original = cv2.calcHist([img_hist], [0], None, [256], [0, 256])

# Aplicar ecualización de histograma para mejorar el contraste
img_ecualizada = cv2.equalizeHist(img_hist)

# Calcular el histograma de la imagen ya ecualizada
hist_ecualizado = cv2.calcHist([img_ecualizada], [0], None, [256], [0, 256])

# Mostrar imagen original, imagen ecualizada y ambos histogramas
plt.figure(figsize=(14, 10))

plt.subplot(2, 2, 1)
plt.imshow(img_hist, cmap='gray')
plt.title('Imagen original (bajo contraste)')
plt.axis('off')

plt.subplot(2, 2, 2)
plt.imshow(img_ecualizada, cmap='gray')
plt.title('Imagen ecualizada')
plt.axis('off')

plt.subplot(2, 2, 3)
plt.plot(hist_original, color='black')
plt.title('Histograma original')
plt.xlabel('Intensidad de gris')
plt.ylabel('Número de píxeles')

plt.subplot(2, 2, 4)
plt.plot(hist_ecualizado, color='black')
plt.title('Histograma ecualizado')
plt.xlabel('Intensidad de gris')
plt.ylabel('Número de píxeles')

plt.tight_layout()
plt.show()

> Primero se obtuvo el histograma de la imagen original mediante `cv2.calcHist`, especificando el canal único (`[0]`), sin máscara, con 256 bins correspondientes a cada nivel de intensidad posible (0-255). En este histograma se observa que los valores de intensidad de la imagen original están concentrados en un rango reducido, lo cual se refleja visualmente como una imagen de bajo contraste.

> Posteriormente se aplicó `cv2.equalizeHist`, función que redistribuye los niveles de intensidad de la imagen de manera que el histograma resultante se aproxime a una distribución uniforme. Al comparar ambos histogramas, se aprecia cómo la ecualización expande el rango dinámico de la imagen, distribuyendo la información en todo el espectro de 0 a 255 y logrando así una mejora visible en el contraste de la imagen médica.

# **Conclusión grupal**

> [Espacio para la conclusión grupal del equipo: describir de manera integradora lo observado en las transformaciones geométricas (traslación, rotación y escala) y en la ecualización de histograma, resaltando la utilidad de estas técnicas en el procesamiento de imágenes médicas.]